# Simply fine-tuning: Qwen2.5-7B-Instruct + QLoRA

Train an adapter to propose one JSON action from a full Simply observation. Start with a three-step smoke test; then start a fresh runtime from the original base for a one-epoch pilot.

**Checkpoint:** `unsloth/Qwen2.5-7B-Instruct-bnb-4bit`, a quantized derivative of `Qwen/Qwen2.5-7B-Instruct`, the checkpoint family used by Simply's prepared hosted Qwen baseline. The hosted OpenRouter/Phala endpoint and local 4-bit model can differ in revision, quantization, serving, templates and decoding; do not assume bit-identical results. For an adapter-effect comparison, run this same local base before and after tuning under matched conditions.

**Data:** 234 training, 58 validation and 60 test decisions from six synthetic mission groups. The rule-teacher data is a small workflow demo, not human behavior or model-performance evidence.

**Status:** this Qwen2.5 adaptation has not been executed on a GPU, nor have its tokenizer and memory requirements been verified in Colab. Follow the checks below before training. No trained adapter or outcome is included.


## Setup

1. Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**,
   and upload this `.ipynb`.
2. Select **Runtime → Change runtime type → Python 3 → GPU**.
   A T4 is a reasonable first attempt; this dataset needs long context, so memory fit
   and runtime are not guaranteed. An L4/A100 gives more headroom if available.
3. Download `simply_qwen_finetuning_demo.zip` from the Simply project onto your computer.
   The upload cell below will ask for that zip.
4. Run cells in order. Read the short note above each cell. Keep `TRAINING_MODE="smoke"`
   on your first attempt. Do not repeatedly rerun the training cell on the same model.

Colab can disconnect and GPU availability varies. Download the adapter at the end.
This notebook saves locally in the Colab session and does not publish to Hugging Face.

### 1. Install Unsloth

Use a fresh GPU runtime. This installs the current dependency-compatible Unsloth stack; resolved versions are saved with your adapter. If Colab asks you to restart, restart the session and resume at step 2. If installation fails, consult the [Unsloth installation guide](https://unsloth.ai/docs/get-started/install-and-update) in a fresh runtime. Avoid mixing installation recipes.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "unsloth", "datasets", "peft", "accelerate", "bitsandbytes",
    "sentencepiece", "protobuf",
])
print("Installation complete. If Colab requests a restart, restart and continue at step 2.")

### 2. Check GPU and choose the first experiment

Import Unsloth before Transformers. `smoke` runs only three optimizer steps with
batch size 1. It verifies the workflow; it is not a meaningful quality experiment.
For a pilot later, use a **fresh model/session** and change only `TRAINING_MODE` to
`"pilot"`. That runs one pass through the 234 training decisions with an effective
batch size of 8. More epochs are not a substitute for more diverse examples.

In [ ]:
from unsloth import FastLanguageModel
import torch
import gc
import hashlib
import importlib.metadata
import inspect
import io
import json
import math
import statistics
import time
import zipfile
from collections import Counter
from pathlib import Path

assert torch.cuda.is_available(), "Choose a GPU runtime in Runtime → Change runtime type."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}; memory: {gpu.total_memory / 2**30:.1f} GiB")

TRAINING_MODE = "smoke"  # First run: "smoke". Later, from a fresh base: "pilot".
assert TRAINING_MODE in {"smoke", "pilot"}
MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 16384
LORA_RANK = 16
LORA_ALPHA = 32
LEARNING_RATE = 1e-4
SEED = 3407
MAX_NEW_TOKENS = 128
RUN_FINAL_TEST = False  # Leave false while choosing settings.
RUN_MEMORY_CHECK = True
RUN_DIRECTORY = Path(f"/content/simply_qwen25_7b_{TRAINING_MODE}_{time.strftime('%Y%m%d_%H%M%S')}")
RUN_DIRECTORY.mkdir(parents=True, exist_ok=False)
print("Experiment:", TRAINING_MODE, "| Results:", RUN_DIRECTORY)

## Steps

### 3. Upload the dataset zip

Select **`simply_qwen_finetuning_demo.zip`** when prompted. This reads only the
three training/evaluation files and their metadata from the archive. It does not
add synthetic fixtures to live Simply history.

In [ ]:
from google.colab import files

uploaded = files.upload()
archives = [(name, content) for name, content in uploaded.items() if name.endswith(".zip")]
assert len(archives) == 1, "Upload exactly one simply_qwen_finetuning_demo.zip file."
archive_name, archive_bytes = archives[0]
required = ["train.jsonl", "validation.jsonl", "test.jsonl", "manifest.json", "provenance.jsonl"]
DATA_DIRECTORY = RUN_DIRECTORY / "data"
DATA_DIRECTORY.mkdir()
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for name in required:
        matches = [item for item in archive.namelist()
                   if item == name or item.endswith("/" + name)]
        assert len(matches) == 1, f"Archive must contain exactly one {name}"
        (DATA_DIRECTORY / name).write_bytes(archive.read(matches[0]))
del uploaded, archives, archive_bytes

### 4. Verify format, labels and split boundaries

The notebook checks the saved file hashes, action JSON and original mission split.
Physical transitions were validated when the demo was generated by Simply; the
Colab checks below preserve that provenance but do not run the simulation engine.
This bundle is intentionally fixed. Expanded datasets need their own manifest and
the same kind of validation rather than turning these assertions off.

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]

manifest = json.loads((DATA_DIRECTORY / "manifest.json").read_text())
assert manifest["synthetic"] and manifest["model_requests"] == 0
for name in ["train.jsonl", "validation.jsonl", "test.jsonl", "provenance.jsonl"]:
    actual_hash = hashlib.sha256((DATA_DIRECTORY / name).read_bytes()).hexdigest()
    assert actual_hash == manifest["files"][name], f"Changed file: {name}"

raw_data = {split: read_jsonl(DATA_DIRECTORY / f"{split}.jsonl")
            for split in ("train", "validation", "test")}
provenance = read_jsonl(DATA_DIRECTORY / "provenance.jsonl")
meta_index = {(row["split"], row["line"]): row for row in provenance}
assert len(meta_index) == len(provenance) == sum(map(len, raw_data.values()))
groups, prompt_hashes = {}, {}

def observation(prompt):
    return json.loads(prompt.split("Current state: ", 1)[1].split("\nChoose one step:", 1)[0])

for split, rows in raw_data.items():
    assert len(rows) == manifest["examples"][split]
    groups[split], prompt_hashes[split] = set(), set()
    for line, row in enumerate(rows, 1):
        assert [m["role"] for m in row["messages"]] == ["user", "assistant"]
        prompt, target = [m["content"] for m in row["messages"]]
        action = json.loads(target)
        assert set(action) == {"step", "product_id", "reason"}
        assert all(isinstance(value, str) for value in action.values())
        assert action["step"] in observation(prompt)["allowed_steps"]
        meta = meta_index[split, line]
        assert meta["synthetic"] and meta["core_validated"]
        assert hashlib.sha256(prompt.encode()).hexdigest() == meta["prompt_sha256"]
        assert hashlib.sha256(target.encode()).hexdigest() == meta["target_sha256"]
        groups[split].add(meta["mission_group"])
        prompt_hashes[split].add(meta["prompt_sha256"])
    print(split, len(rows), "examples | mission groups:", sorted(groups[split]))
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert not groups[left] & groups[right], "Mission leakage"
    assert not prompt_hashes[left] & prompt_hashes[right], "Prompt leakage"
print("Checks passed. Both layouts/catalog are shared across splits; these are tiny demo holdouts.")

### 5. Measure token lengths before loading model weights

The existing observations are long. This cell measures each split with the actual Qwen2.5 tokenizer and checks the complete example plus generation allowance fits the selected context. It stops instead of silently truncating. Lengths measured for Qwen3 do not establish Qwen2.5 token lengths. Inspect the printed unmasked target and preserve this same template during local base-versus-adapter inference.


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
reference_template = tokenizer.chat_template

def tokenize_example(row):
    messages = row["messages"]
    full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
    prefix_ids = tokenizer.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True)
    assert full_ids[:len(prefix_ids)] == prefix_ids, "Unexpected chat-template prefix mismatch."
    labels = [-100] * len(prefix_ids) + full_ids[len(prefix_ids):]
    assert any(label != -100 for label in labels), "No assistant tokens to learn from."
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}

tokenized_rows = {split: [tokenize_example(row) for row in rows]
                  for split, rows in raw_data.items()}
length_summary = {}
for split, rows in tokenized_rows.items():
    lengths = [len(row["input_ids"]) for row in rows]
    length_summary[split] = {"count": len(rows), "min": min(lengths),
                             "median": statistics.median(lengths), "max": max(lengths)}
    print(split, length_summary[split])
    assert max(lengths) + MAX_NEW_TOKENS <= MAX_SEQ_LENGTH, (
        f"{split} needs more context. Increase MAX_SEQ_LENGTH and use sufficient GPU memory; "
        "do not truncate the observations or answers."
    )
tokenized_data = {split: Dataset.from_list(rows) for split, rows in tokenized_rows.items()}
print("Model revision:", MODEL_REVISION)
print("Learned text in the first example:")
print(tokenizer.decode([value for value in tokenized_rows["train"][0]["labels"] if value != -100]))

### 6. Load the frozen 4-bit model

`load_in_4bit=True` reduces base-model memory; `full_finetuning=False` preserves
its weights. LoRA adapters are added after the baseline measurement. The 16K
context is a data requirement, not a promise that every Colab GPU can train it.

In [ ]:
model, model_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    revision=MODEL_REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
)
model_tokenizer.chat_template = reference_template
model_tokenizer.pad_token = tokenizer.pad_token
assert model_tokenizer.apply_chat_template(raw_data["train"][0]["messages"], tokenize=True) == tokenized_rows["train"][0]["input_ids"]
tokenizer = model_tokenizer
print("Model loaded. Base model remains frozen during QLoRA training.")

## Checks

### 7. Define a fixed, small validation comparison

Select one validation example for each available action type **before seeing model
outputs**. Compare the untouched and tuned model on those same observations with
the same decoding settings. This is a small diagnostic, not whole-trip evaluation.

Strict JSON rate checks the response contract. Observation-valid action rate checks
the public legal moves, shelf reach and checkout conditions. Teacher action agreement
compares `step` and `product_id`; a different valid route can disagree with this
teacher. None of these checks establishes trip completion or human realism.

In [ ]:
def parse_and_check(text, prompt):
    try:
        action = json.loads(text)
    except (ValueError, TypeError):
        return None, False
    if not isinstance(action, dict) or set(action) != {"step", "product_id", "reason"}:
        return None, False
    if not all(isinstance(value, str) for value in action.values()):
        return None, False
    facts = observation(prompt)
    step, product_id = action["step"], action["product_id"]
    owned = {item["product_id"] for item in facts["cart"]}
    valid = step in facts["allowed_steps"]
    if step.startswith("move_"):
        valid = valid and not product_id and step in facts["legal_move_destinations"]
    elif step == "grab":
        valid = valid and product_id in {p["product_id"] for p in facts["grabbable_products"]} and product_id not in owned
    elif step == "remove":
        valid = valid and product_id in owned
    elif step == "checkout":
        valid = valid and not product_id and bool(owned) and facts["position"] in facts["checkout_cells"]
    else:
        valid = False
    return action, bool(valid)

selected_indices = []
seen_steps = set()
for index, row in enumerate(raw_data["validation"]):
    step = json.loads(row["messages"][-1]["content"])["step"]
    if step not in seen_steps:
        seen_steps.add(step)
        selected_indices.append(index)
comparison_rows = [raw_data["validation"][index] for index in selected_indices]
print("Fixed validation rows (1-based):", [index + 1 for index in selected_indices])
print("Action types:", sorted(seen_steps))

def compare_on_rows(rows, label):
    FastLanguageModel.for_inference(model)
    model.eval()
    records = []
    for row in rows:
        prompt = row["messages"][0]["content"]
        expected = json.loads(row["messages"][1]["content"])
        rendered = tokenizer.apply_chat_template(row["messages"][:-1], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(rendered, return_tensors="pt", add_special_tokens=False).to(model.device)
        assert inputs["input_ids"].shape[1] + MAX_NEW_TOKENS <= MAX_SEQ_LENGTH
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            output = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            )
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        generated = output[0, inputs["input_ids"].shape[1]:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        actual, valid = parse_and_check(text, prompt)
        agreement = bool(actual and (actual["step"], actual["product_id"]) == (expected["step"], expected["product_id"]))
        records.append({"raw_response": text, "strict_schema": actual is not None,
                        "observation_valid": valid, "teacher_action_agreement": agreement,
                        "expected": expected, "seconds": elapsed})
        del inputs, output, generated
    result = {"label": label, "examples": len(records),
              "strict_schema_rate": sum(r["strict_schema"] for r in records) / len(records),
              "observation_valid_rate": sum(r["observation_valid"] for r in records) / len(records),
              "teacher_action_agreement_rate": sum(r["teacher_action_agreement"] for r in records) / len(records),
              "mean_seconds_per_decision": statistics.mean(r["seconds"] for r in records),
              "records": records}
    (RUN_DIRECTORY / f"{label}.json").write_text(json.dumps(result, indent=2))
    print(json.dumps({key: value for key, value in result.items() if key != "records"}, indent=2))
    return result

### 8. Measure the original model

This performs a few local GPU generations, with no paid model API calls. The first
generation can include compilation overhead, so latency is only a rough diagnostic.
Greedy decoding is a fixed comparison choice for this experiment, not a universal
recommendation for all Qwen tasks.

In [ ]:
baseline = compare_on_rows(comparison_rows, "baseline_validation")

### 9. Add LoRA adapters and verify which weights can change

Rank 16 / alpha 32 / learning rate 1e-4 are starting experiment settings. They are
not tuned optima. We adapt attention and feed-forward projections, keep dropout
at zero, and enable Unsloth gradient checkpointing to reduce memory use.

In [ ]:
FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()
trainable = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
assert trainable and all("lora_" in name for name in trainable), "Unexpected trainable base weights."

### 10. Check the longest example fits before committing to a run

This performs one forward/backward pass without an optimizer update, then clears
the gradients. If memory runs out, use a larger-memory runtime or work on a shorter
prompt design that is used consistently in training and the eventual Simply provider.
Do not remove map/state fields only from training, silently drop long examples, or
cut off the answer. Colab Free may be limiting even with a small model.

In [ ]:
from transformers import DataCollatorForSeq2Seq

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding=True, pad_to_multiple_of=8,
    label_pad_token_id=-100, return_tensors="pt",
)
if RUN_MEMORY_CHECK:
    longest = max(tokenized_rows["train"], key=lambda row: len(row["input_ids"]))
    batch = collator([longest])
    batch = {key: value.to(model.device) for key, value in batch.items()}
    FastLanguageModel.for_training(model)
    model.train()
    model.zero_grad(set_to_none=True)
    try:
        autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        with torch.autocast(device_type="cuda", dtype=autocast_dtype):
            memory_loss = model(**batch).loss
        memory_loss.backward()
        print("Longest example backward pass passed:", len(longest["input_ids"]), "tokens")
    except torch.OutOfMemoryError as error:
        raise RuntimeError(
            "The full observation does not fit this GPU for training. Restart with more GPU memory "
            "or redesign a shorter, consistent runtime/training prompt. Do not truncate the answer."
        ) from error
    finally:
        model.zero_grad(set_to_none=True)
        del batch
        if "memory_loss" in globals():
            del memory_loss
        gc.collect()
        torch.cuda.empty_cache()

### 11. Configure supervised training

The examples are already tokenized with **user tokens masked to -100**, so only
assistant action tokens contribute to the loss. Transformers Trainer runs SFT on
the Unsloth model; this avoids an extra SFTTrainer tokenization/truncation pass.
The collator pads labels with -100 too. No packing is used for the first experiment.

`smoke`: 3 optimizer steps, batch 1, accumulation 1. `pilot`: 1 epoch, batch 1,
accumulation 8. A pilot evaluates validation loss every 10 optimizer steps and
retains the best saved checkpoint by that loss. Validation loss alone is not the
selection criterion for deploying an agent.

In [ ]:
from transformers import Trainer, TrainingArguments, set_seed

set_seed(SEED)
is_pilot = TRAINING_MODE == "pilot"
arguments = TrainingArguments(
    output_dir=str(RUN_DIRECTORY / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8 if is_pilot else 1,
    num_train_epochs=1,
    max_steps=-1 if is_pilot else 3,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    eval_strategy="steps" if is_pilot else "no",
    eval_steps=10,
    prediction_loss_only=True,
    save_strategy="steps" if is_pilot else "no",
    save_steps=10,
    save_total_limit=2,
    load_best_model_at_end=is_pilot,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    remove_unused_columns=False,
)
trainer_options = dict(
    model=model, args=arguments,
    train_dataset=tokenized_data["train"], eval_dataset=tokenized_data["validation"],
    data_collator=collator,
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_options["processing_class"] = tokenizer
else:
    trainer_options["tokenizer"] = tokenizer
trainer = Trainer(**trainer_options)
sample_batch = collator([tokenized_rows["train"][0]])
assert (sample_batch["labels"] != -100).any()
print("Training mode:", TRAINING_MODE, "| Number of training examples:", len(tokenized_data["train"]))

### 12. Train once

Read the first few logs. The loss should be finite; a nonzero loss does not mean
something is wrong. Stop if it becomes NaN. Loss falling means the model is learning
the supplied teacher labels; it does not establish better trips or generalization.
Long full observations can make even this small dataset slow on a T4.

In [ ]:
assert not globals().get("TRAINING_FINISHED", False), "Start a fresh base model for another experiment."
FastLanguageModel.for_training(model)
model.train()
training_result = trainer.train()
TRAINING_FINISHED = True
print(training_result.metrics)
(RUN_DIRECTORY / "training_metrics.json").write_text(json.dumps(training_result.metrics, indent=2))
(RUN_DIRECTORY / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2))

### 13. Compare the tuned model on the exact same validation examples

A smoke test may show no improvement. On this tiny, correlated holdout, even an
apparent improvement is preliminary. Review the saved raw responses, especially
grabs and checkout. Different valid moves can disagree with the deterministic teacher.

In [ ]:
tuned = compare_on_rows(comparison_rows, "tuned_validation")
for metric in ["strict_schema_rate", "observation_valid_rate", "teacher_action_agreement_rate"]:
    print(f"{metric}: baseline={baseline[metric]:.3f}; tuned={tuned[metric]:.3f}")
print("Example tuned response:", tuned["records"][0]["raw_response"])

### 14. Optional: run the reserved test split after settings are fixed

Keep `RUN_FINAL_TEST=False` while choosing data, epochs and learning rate. When
your choices are final, set it to True and run this cell. It evaluates all 60 test
decisions. Repeatedly choosing settings from test results turns this into another
validation set; create a new holdout for the next experiment.

In [ ]:
if RUN_FINAL_TEST:
    final_test = compare_on_rows(raw_data["test"], "final_test")
else:
    print("Test split left unused. Change RUN_FINAL_TEST only after fixing experiment settings.")

### 15. Save and download the adapter

This saves **LoRA adapter weights + tokenizer + experiment metadata**, not a complete
standalone 7-billion-parameter model. Loading it later also needs the exact base
checkpoint recorded in `experiment.json`. A smoke adapter is labeled as a smoke
artifact. Download it before the Colab session ends.

The experiment records the model revision, tokenizer template, file hashes and
package versions. There is no Hugging Face publishing or Simply provider change.

In [ ]:
ADAPTER_DIRECTORY = RUN_DIRECTORY / "adapter"
model.save_pretrained(str(ADAPTER_DIRECTORY))
tokenizer.save_pretrained(str(ADAPTER_DIRECTORY))
package_names = ["unsloth", "unsloth_zoo", "transformers", "datasets", "peft", "accelerate", "bitsandbytes", "torch"]
versions = {}
for package in package_names:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "unavailable"
experiment = {
    "project": "Simply", "training_mode": TRAINING_MODE, "method": "QLoRA_SFT",
    "base_model": MODEL_ID, "base_revision": MODEL_REVISION,
    "max_seq_length": MAX_SEQ_LENGTH, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE, "seed": SEED,
    "assistant_output_only_loss": True, "load_in_4bit": True,
    "training_arguments": arguments.to_dict(), "packages": versions,
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
    "dataset": manifest["dataset"], "dataset_counts": manifest["examples"],
    "dataset_hashes": manifest["files"], "token_lengths": length_summary,
    "gpu": gpu.name, "test_used": bool(RUN_FINAL_TEST),
    "limitation": "Synthetic rule-teacher demo; no whole-trip or human-realism evaluation.",
}
(RUN_DIRECTORY / "experiment.json").write_text(json.dumps(experiment, indent=2, default=str))

import shutil
export_zip = shutil.make_archive(str(RUN_DIRECTORY) + "_adapter", "zip", root_dir=RUN_DIRECTORY,
                                 base_dir="adapter")
# Add the small reports, without bundling full checkpoints or dataset copies.
with zipfile.ZipFile(export_zip, "a", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIRECTORY.glob("*.json")):
        archive.write(path, path.name)
print("Saved:", export_zip)
files.download(export_zip)

## Next Steps

1. **After the smoke test:** download its adapter, start a fresh GPU session, change
   `TRAINING_MODE` to `"pilot"`, upload the same data and run one epoch from the original base.
2. **After the pilot:** inspect validation responses. Add examples targeting actual
   failures: checkout urgency, budget tradeoffs, removals, recovery and varied missions.
   This corpus mostly teaches list completion. Do not expand it by duplicating rows.
3. **When settings are fixed:** evaluate the reserved test split once. For meaningful
   project results, reserve more independent shopper missions before generating training data.
4. **Before replacing Luna/Terra:** integrate the adapter into a provider that proposes one action to
   Python core. The existing hosted Qwen2.5 provider does not serve this adapter. Evaluate untouched and tuned Qwen in full trips on the same frozen inputs,
   first without navigation assistance and then with the same assistance policy.
   Measure fulfillment, checkout completion, invalid actions, loops, assistance rate,
   latency and compute costs. Notebook action agreement is not a substitute.
5. **Saving and serving:** use the adapter with its base model and identical chat template.
   Keep the model/checkpoint identity in each new run. Do not relabel historical recordings.
   Consider GGUF/merged exports only after validating the adapter in its training setup.

### Troubleshooting

| Symptom | First check |
|---|---|
| No GPU / CUDA unavailable | Runtime → Change runtime type → GPU |
| Import/dependency error | Fresh runtime; use the current official Unsloth installation cell |
| Out of memory | Keep batch size 1; use more GPU memory for 16K observations; preserve all state/answers |
| All labels are -100 | Stop and fix chat-template/prefix masking before training |
| Wrong products or extra prose | Inspect full observations, exact JSON targets and the saved tokenizer template |
| Training loss drops, validation gets worse | Stop the run; reduce repeated training and improve coverage/labels |
| Good notebook scores, poor full trips | Investigate state distribution shift and evaluate through Python core |
| Colab disconnects | Recover saved checkpoints if still present; otherwise reload base + downloaded adapter |

### References

- [Official Qwen2.5-7B-Instruct model card](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct)
- [Unsloth 4-bit Instruct checkpoint](https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit)
- [Unsloth fine-tuning guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide)
- [LoRA hyperparameters](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide)

The Qwen2.5 notebook adaptation has not been run on a GPU. Record results and package versions before making any training or quality claim.
